# 06 · App de Notas de Campo — Databricks Apps + Lakebase

**Objetivo:** desplegar la app `cfe-notas-obra` (codigo en `apps/cfe-notas-obra` del repo) para que los residentes registren notas de campo sobre las obras, con Lakebase Postgres como base transaccional.

En este laboratorio despliegas la app de **Notas de Campo CFE** usando **solo la interfaz** de Databricks. El catalogo de obras de la capa Gold llega a Lakebase con una **synced table** (paso 4) y la app lo lee directamente desde ahi gracias a dos variables de entorno (`OBRAS_SCHEMA`, `OBRAS_TABLE`) en `app.yaml`.

```
Usuario --> App (app_v3.py) --> Lakebase Postgres (esquema public)
                                  |- synced table    (solo lectura, paso 4)
                                  |- notas_obra      (lo crea la app)
                                  '- bitacora_app    (lo crea la app)
                                          |
            Job: sync_lakebase_to_delta   v   (paso 8)
            Tablas Delta *_sync en Unity Catalog para analitica
```

| Paso | Donde | Que haces |
| --- | --- | --- |
| 1 | Lakebase | Crear el proyecto de Postgres |
| 2 | Apps | Crear la app (sin desplegar) |
| 3 | Apps | Ligar Lakebase como recurso de la app |
| 4 | Catalog Explorer | Sincronizar el catalogo de obras (synced table) |
| 5 | Lakebase SQL Editor | Verificar datos y permisos |
| 6 | Editor de archivos | Completar `app.yaml` |
| 7 | Apps | Desplegar y probar la app |
| 8 | Jobs & Pipelines | Programar la sincronizacion a Delta |

**Antes de empezar**
* Debe existir `gold_fact_obras` (pipeline del lab 03).
* Usa un nombre propio para tus recursos, por ejemplo `cfe-notas-<tus-iniciales>`, para no chocar con otros participantes.
* Ten a la mano un bloc de notas: vas a copiar **4 valores** (endpoint, host, client ID de la app y nombre de la synced table).

> Los nombres de botones y menus pueden variar ligeramente segun la version de la interfaz.

## Paso 1 - Crear el proyecto de Lakebase

1. En el selector de aplicaciones (arriba a la derecha) o en el menu lateral, entra a **Lakebase Postgres**.
2. Clic en **Create project** (Crear proyecto).
3. Nombre: `cfe-notas-<tus-iniciales>`. Version de Postgres: la mas reciente. Deja el resto por defecto.
4. Clic en **Create**. En segundos se crean automaticamente:
   * el branch **production**
   * el compute **primary** (lectura/escritura, con autoescalado y scale-to-zero)
   * la base de datos **databricks_postgres**
5. Dentro del proyecto, abre el branch **production** y clic en **Connect** (Conectar). Anota:
   * **Host** -> ejemplo: `ep-xxxx.database.<region>.azuredatabricks.net` -> sera tu `PGHOST`
   * **Endpoint** -> `projects/<proyecto>/branches/production/endpoints/primary` -> sera tu `LAKEBASE_ENDPOINT`

> Si el instructor ya creo un proyecto compartido (por ejemplo `lakebaseworkshop`), puedes usarlo y saltar este paso.

## Paso 2 - Crear la app (sin desplegar)

1. Menu lateral -> **Compute** -> pestana **Apps** -> **Create app**.
2. Elige **Create a custom app** (app personalizada).
3. Nombre: `cfe-notas-<tus-iniciales>` (minusculas y guiones). Descripcion: *Notas de campo CFE con Lakebase*.
4. Clic en **Create app**. Espera a que el compute de la app quede **Running** (2-4 min).
5. En la pagina de la app, pestana **Authorization** (o en los detalles de la app), anota el **Client ID** del service principal de la app -> sera tu `PGUSER`.

> La app corre con su propio **service principal**: una identidad tecnica, distinta de tu usuario, a la que hay que darle permisos.

## Paso 3 - Ligar Lakebase como recurso de la app

1. En la pagina de la app, clic en **Edit** (Editar).
2. En **App resources**, clic en **Add resource** -> **Database** / **Lakebase**.
3. Selecciona tu proyecto, el branch **production** y la base de datos **databricks_postgres**.
4. Permiso: **Can connect and create**.
5. Guarda los cambios (**Save**).

Con esto Databricks crea en Postgres un **rol para el service principal de la app** y le permite conectarse y crear tablas. No necesitas crear usuarios ni contrasenas: la app usa tokens OAuth temporales.

> Puedes confirmarlo en Lakebase: proyecto -> branch **production** -> **Roles**. Debe aparecer un rol con el Client ID del paso 2.

## Paso 4 - Sincronizar el catalogo de obras a Lakebase (synced table)

La app muestra la lista de obras leyendo una tabla en Lakebase. Usamos una **synced table**: Databricks crea un pipeline gestionado que copia una tabla de Unity Catalog hacia Postgres (*reverse ETL*).

1. Menu lateral -> **Catalog** -> `dipidev` -> `default` -> abre **gold_fact_obras**.
2. Arriba a la derecha: **Create** -> **Synced table**.
3. Configura:
   * **Name:** `gold_fact_obras_<tus-iniciales>` (catalogo `dipidev`, esquema `default`). Debe ser unico: todos comparten el mismo esquema.
   * **Destino:** tu proyecto de Lakebase -> branch **production** -> base de datos **databricks_postgres**.
   * **Primary key:** `IdObra` (es unico: 318 obras, sin nulos).
   * **Sync mode:** en **Advanced settings** elige **Snapshot** (copia completa bajo demanda; suficiente para un catalogo que cambia poco).
4. Clic en **Create**. Espera a que el estado de la synced table indique que la sincronizacion termino (unos minutos la primera vez).

**Resultado:** en Postgres aparece la tabla de solo lectura `"default"."gold_fact_obras_<tus-iniciales>"` (el esquema de Unity Catalog se vuelve el esquema de Postgres). La app lee directamente de esta tabla gracias a las variables `OBRAS_SCHEMA` y `OBRAS_TABLE` en `app.yaml` (paso 6).

5. Anota el **nombre exacto** de la synced table (ej. `gold_fact_obras_jg`) -> sera tu `OBRAS_TABLE` en el paso 6.

> Para refrescar el catalogo despues (por ejemplo, tras correr el pipeline del lab 03), abre la synced table y clic en **Sync now**.

## Paso 5 - Verificar datos y permisos (Lakebase SQL Editor)

1. Ve a **Lakebase Postgres** -> tu proyecto -> branch **production** -> **SQL Editor**.
2. Confirma que la synced table tiene datos (reemplaza `<tus-iniciales>`):

```sql
SELECT "IdObra", "NombreObra" FROM "default"."gold_fact_obras_<tus-iniciales>" ORDER BY "IdObra" LIMIT 10;
```

> Las comillas dobles son obligatorias: `default` es palabra reservada y los nombres de columna conservan mayusculas.

3. La app necesita crear sus tablas `notas_obra` y `bitacora_app` en el esquema `public`. El recurso de Lakebase del paso 3 le da acceso general, pero si encuentras errores de permisos, ejecuta:

```sql
GRANT USAGE, CREATE ON SCHEMA public TO "<CLIENT_ID>";
GRANT USAGE ON SCHEMA "default" TO "<CLIENT_ID>";
GRANT SELECT ON ALL TABLES IN SCHEMA "default" TO "<CLIENT_ID>";
```

> Reemplaza `<CLIENT_ID>` por el Client ID de la app (paso 2), **conservando las comillas dobles**.

## Paso 6 - Completar `app.yaml`

`app.yaml` le indica a la app que programa ejecutar y como conectarse a Lakebase.

1. En el explorador de archivos (panel izquierdo), abre la carpeta `apps/cfe-notas-obra` del repo (un nivel arriba de `notebooks/`) y luego el archivo **app.yaml**.
2. Deja su contenido asi, con **tus** valores de los pasos 1, 2 y 4:

```yaml
command:
  - python
  - app_v3.py

env:
  - name: LAKEBASE_ENDPOINT
    value: "projects/<tu-proyecto>/branches/production/endpoints/primary"
  - name: PGHOST
    value: "<host del paso 1>"
  - name: PGPORT
    value: "5432"
  - name: PGDATABASE
    value: "databricks_postgres"
  - name: PGUSER
    value: "<CLIENT_ID de la app del paso 2>"
  - name: OBRAS_SCHEMA
    value: "default"
  - name: OBRAS_TABLE
    value: "<nombre de tu synced table del paso 4>"
```

> `OBRAS_TABLE` es el nombre **sin esquema** de la synced table (ej. `gold_fact_obras_jg`). `OBRAS_SCHEMA` es `default` si tu esquema de Unity Catalog es `default`.

3. Los cambios se guardan solos. Revisa que no queden `<` `>` en el archivo.

## Paso 7 - Desplegar y probar la app

**Desplegar**
1. Ve a **Compute** -> **Apps** -> tu app.
2. Clic en **Deploy** (Desplegar).
3. En *Source code path* selecciona la carpeta del repo `.../workshopcfe/apps/cfe-notas-obra`.
4. Confirma con **Deploy**. En la pestana **Deployments** el estado debe pasar a **Succeeded** (1-2 min).
5. Si algo falla, revisa la pestana **Logs**.

**Probar**
1. Abre la **URL** de la app (aparece en la parte superior de su pagina).
2. Selecciona una obra, escribe una nota y clic en **Registrar Nota**.
3. En la tabla de notas recientes cambia el estado: **Abierta -> En Proceso -> Resuelta**.

**Confirmar en Lakebase** (SQL Editor del paso 5):

```sql
SELECT id, id_obra, tipo_nota, prioridad, estado, usuario, fecha_registro
FROM notas_obra ORDER BY id DESC LIMIT 10;

SELECT accion, detalle, fecha FROM bitacora_app ORDER BY id DESC LIMIT 10;
```

Las tablas `notas_obra` y `bitacora_app` las creo la app en su primer uso.

## Paso 8 - Sincronizar Lakebase -> Delta (Jobs)

Las notas viven en Lakebase (transaccional). Para analizarlas en dashboards, Genie o el modelo de riesgo, el notebook `sync_lakebase_to_delta` las copia a tablas Delta `notas_obra_sync` y `bitacora_app_sync`.

**Ajustar el notebook de sync**
1. Abre `sync_lakebase_to_delta` en `apps/cfe-notas-obra`.
2. Cambia `LAKEBASE_ENDPOINT` y `PGHOST` por tus valores del paso 1. Si tu catalogo/esquema no es `dipidev.default`, ajusta tambien `CATALOG` y `SCHEMA`.

**Crear el job**
1. Menu lateral -> **Jobs & Pipelines** -> **Create** -> **Job**.
2. Nombre: `CFE - Sync Lakebase a Delta (<tus-iniciales>)`.
3. Tarea: tipo **Notebook**, ruta `.../apps/cfe-notas-obra/sync_lakebase_to_delta`, compute **Serverless**.
4. En el panel derecho, **Add trigger** -> **Scheduled** -> cada **1 hora**.
5. Clic en **Run now** y espera a que termine en verde.

**Verificar en Catalog Explorer**
* Abre **Catalog** -> `dipidev` -> `default` y busca `notas_obra_sync`. En **Sample Data** deben verse las notas que registraste.

## Troubleshooting

| Sintoma en la app | Causa probable | Solucion |
| --- | --- | --- |
| Lista de obras con `password authentication failed` | La app no tiene rol en Lakebase o `PGUSER` es incorrecto | Revisar paso 3 y el Client ID en `app.yaml` |
| `permission denied for table ...` o `permission denied for schema ...` | Faltan permisos en Lakebase | Ejecutar los GRANT del paso 5 y re-desplegar |
| `relation "..." does not exist` o lista de obras vacia | `OBRAS_TABLE` incorrecto en `app.yaml` o la synced table no termino | Revisar el nombre en Catalog, verificar en el paso 5, re-desplegar |
| La synced table no termina o falla | Error en el pipeline gestionado | Abre la synced table en Catalog y revisa su estado / pipeline |
| Deploy en estado *Failed* | Error en `app.yaml` o en el codigo | Pestana **Logs** de la app |
| La primera carga tarda unos segundos | El compute de Lakebase escalo a cero | Es normal; despierta solo |

## Limpieza (al terminar el workshop)

* **Detener la app**: pagina de la app -> **Stop** (deja de consumir compute).
* **Pausar el job** de sync: Jobs & Pipelines -> tu job -> pausar el trigger.
* **Borrar la synced table** `gold_fact_obras_<tus-iniciales>` desde Catalog (elimina tambien su pipeline).
* **Borrar el proyecto de Lakebase** (opcional): elimina todas las notas de forma permanente.